# Lab 5: Configure an AI/BI Genie Space — TTC Open Data

This lab practices configuring **AI/BI Genie** so business users could ask natural-language questions about the TTC data built in Lab 3 (`ttc_open_data.bronze.raw_routes`, `ttc_open_data.silver.ttc_complaints`, `ttc_open_data.silver.vw_complaints_with_route`, `ttc_open_data.gold.route_complaint_summary`).

Genie space configuration happens almost entirely in the **Databricks UI**, not in notebook cells — so this notebook mixes a scriptable SQL prep step with a step-by-step guide for the parts you do yourself in the Genie UI. Where a task has a concrete right answer (SQL expression syntax, example query), it's given as a `# TODO`-style task for you to write, not a filled-in answer.

> Complete **Lab 3** first — this lab configures Genie over those tables/views.

## Exercise 1 (scriptable): Enrich Column Descriptions Before Building the Space

**Concept:** Genie inherits Unity Catalog's column descriptions as a starting point for its knowledge store — better descriptions now means less manual work in the Genie UI later.

**Task:** Using `ALTER TABLE ... ALTER COLUMN ... COMMENT '...'`, add descriptive comments to:
- `ttc_open_data.silver.ttc_complaints`: `issue_type`, `filed_date`, `resolved`
- `ttc_open_data.bronze.raw_routes`: `route_short_name`, `route_long_name`, `route_type`

Write comments a non-technical business user would find helpful — e.g. explain what the GTFS `route_type` codes (0/1/3) actually mean, since Genie can't infer that on its own.

> 🤖 **Genie Code tip:** *"Write ALTER TABLE ALTER COLUMN COMMENT statements in Databricks SQL to add business-friendly descriptions to several columns across two tables."*

In [ ]:
%sql
-- TODO: ALTER TABLE ttc_open_data.silver.ttc_complaints ALTER COLUMN issue_type COMMENT '...'
-- TODO: ALTER TABLE ttc_open_data.silver.ttc_complaints ALTER COLUMN filed_date COMMENT '...'
-- TODO: ALTER TABLE ttc_open_data.silver.ttc_complaints ALTER COLUMN resolved COMMENT '...'

-- TODO: ALTER TABLE ttc_open_data.bronze.raw_routes ALTER COLUMN route_short_name COMMENT '...'
-- TODO: ALTER TABLE ttc_open_data.bronze.raw_routes ALTER COLUMN route_long_name COMMENT '...'
-- TODO: ALTER TABLE ttc_open_data.bronze.raw_routes ALTER COLUMN route_type COMMENT '...' (explain the GTFS codes: 0 = Streetcar, 1 = Subway, 3 = Bus)

## Exercise 2 (UI): Create the Genie Space

**Concept:** A Genie space is scoped to a small, focused set of tables — Databricks recommends 5 or fewer. More tables means more ambiguity for Genie to resolve, not more capability.

**Task:**
1. In the Databricks workspace sidebar, select **Genie**.
2. Select **New Genie space**.
3. Name it something like `TTC Complaints Analytics`.
4. Add these tables/views:
   - `ttc_open_data.bronze.raw_routes`
   - `ttc_open_data.silver.vw_complaints_with_route`
   - `ttc_open_data.gold.route_complaint_summary`
5. Select **Create**.

## Exercise 3 (UI): Configure the Knowledge Store

**Concept:** The knowledge store is where you teach Genie business vocabulary — table/column descriptions, synonyms, and which columns to hide from users entirely.

**Task:** In your Genie space, select **Configure > Data** and:
1. Write a description for `vw_complaints_with_route` explaining, in plain language, what a row in this view represents.
2. On the `issue_type` column, add at least 3 synonyms a rider or manager might actually type (think: "problem", "category", "complaint type").
3. On `route_short_name`, add synonyms like "route number" or "route".
4. Write a description for `route_complaint_summary` explaining that it's precomputed and refreshes on a schedule (this matters — Genie should tell users the data isn't always up-to-the-second).
5. Check for any columns that wouldn't mean anything to a business user (internal IDs, etc.) and use **Hide selected columns** from the Actions menu if you find any.

## Exercise 4 (UI): Define Relationships (Joins)

**Concept:** Genie needs explicit join relationships to combine tables correctly when a question spans more than one. Since `vw_complaints_with_route` is already pre-joined, this step mainly matters if you add the raw tables separately instead of (or in addition to) the view.

**Task:** If you added `bronze.raw_routes` alongside the silver/gold objects, go to **Configure > Data > Joins** and define the join between `ttc_complaints.route_id` and `raw_routes.route_id`. Think about which relationship type is correct — one route has many complaints, so is this One to many, Many to one, or Many to many from the complaints side? Set it accordingly.

## Exercise 5 (UI): Add SQL Expressions and Example Queries

**Concept:** SQL expressions define reusable business measures/dimensions Genie can reference by name instead of re-deriving them each time. Example queries (especially parameterized, "Trusted" ones) anchor Genie's SQL generation to a known-correct pattern.

**Task — SQL Expressions** (go to **Configure > Instructions > SQL Expressions**):
1. Add a **measure** named `unresolved_complaint_count`. Write the expression yourself — it should count complaints where `resolved = false`, using a `COUNT(CASE WHEN ... THEN 1 END)` pattern.
2. Add a **dimension** named `complaint_month`. Write the expression using `DATE_TRUNC` to bucket `filed_date` by month.

**Task — Example Queries** (go to the **SQL Queries** tab):
1. Add an example for the question *"What's our current complaint volume by route?"* — write the SQL yourself: group `vw_complaints_with_route` by `route_short_name` and `route_long_name`, count rows, order by count descending.
2. Add a second, **parameterized** example for *"How many unresolved complaints were filed for route :route_short_name?"* — use the `:route_short_name` parameter syntax in a `WHERE` clause alongside `resolved = false`, and mark it **Trusted**.

> 🤖 **Genie Code tip:** *"Write a COUNT with a CASE WHEN condition for a Genie SQL expression measure, a DATE_TRUNC expression for a monthly dimension, and a parameterized SQL query using a :parameter_name placeholder in a WHERE clause."*

## Exercise 6 (UI): Test the Space

**Task:** Ask Genie these questions and compare its answers/generated SQL against what you'd expect. For each one, note whether it used the SQL expressions or example queries you just configured:

1. "Which route has the most complaints?"
2. "How many complaints are about overcrowding?"
3. "How many unresolved complaints were filed for route 504?" (should exercise your Trusted parameterized example)
4. "Show me complaint counts by month." (should exercise your `complaint_month` dimension)
5. "Which issue types are most common on subway routes vs. streetcar routes?" (tests whether Genie correctly uses `route_type` — only works well if your Exercise 1 comment explained the GTFS codes)

## Exercise 7 (UI): Iterative Refinement

**Task:**
1. For any question from Exercise 6 where Genie's answer looks wrong, select **Show generated code** to see the SQL it wrote. Figure out *why* it's wrong — usually a missing synonym, a missing join, or an ambiguous column description.
2. Fix the SQL if needed, then select **Add as instruction** so Genie learns from the correction.
3. Rate a few responses using the thumbs up/down feedback mechanism, then check the **Monitoring** tab to see how that feedback surfaces.
4. Write down which of your 5 test questions worked on the first try and which needed a fix — that's your signal for what still needs work in the knowledge store.

## Recap

By the end of this notebook you should have practiced: enriching Unity Catalog column comments to seed Genie's knowledge store, building a Genie space over the TTC complaint/route data, table and column descriptions with synonyms, defining a join relationship, writing a SQL expression measure and dimension, writing two example SQL queries (one parameterized/Trusted), and a full test-and-refine cycle using real natural-language questions.

This closes out the Unit 5 lab set: **Lab 3** (Unity Catalog objects, functions, procedures & permissions), **Lab 4** (foreign catalog over BigQuery), **Lab 5** (Genie space) — all built on the same TTC open data used since Lab 1.